# Actin / Viractin Analysis — automated pipeline

Project layout:

```text
project/
├── marine_metagenome_diversity_pipeline.ipynb
├── actin_viractin_pipeline.ipynb
├── data/
│   ├── diversity/
│   └── actin/
│       ├── 01_SUMMARY_NCLDV_Genomes_with_.../
│       ├── data_Actin/
│       ├── tree_actin/
│       ├── Amino_acid_sequences_EL-actin_reference_clade.fa
│       └── QEE15133_viruses_blastp.csv
└── results/
    ├── diversity/
    └── actin/
        ├── work/
        └── logs/
```

The notebook follows the TP workflow using the **non-Anvi'o route**:

```text
BlastP
→ Prodigal
→ MAFFT
→ HMMER
→ SeqKit
→ MAFFT
→ trimAl
→ IQ-TREE
→ iTOL
```

The setup searches recursively inside `data/actin/`, so the downloaded folders do not need to be reorganized.


## 0. Setup

In [1]:
from pathlib import Path
import subprocess, shutil, re
import pandas as pd
from IPython.display import display, FileLink

ROOT = Path.cwd()
DATA = ROOT / "data" / "actin"
RESULTS = ROOT / "results" / "actin"
WORK = RESULTS / "work"
LOGS = RESULTS / "logs"

WORK.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

# Find the required files automatically
ACTIN_ARPS = next(DATA.rglob("Actin_ARPs_fasta.fa"), None)
REFERENCE = next(
    DATA.rglob("Amino_acid_sequences_EL-actin_reference_clade.fa"),
    None
)
BIN_BY_BIN = next(
    (p for p in DATA.rglob("bin_by_bin") if p.is_dir()),
    None
)

if ACTIN_ARPS is None:
    raise FileNotFoundError("Could not find Actin_ARPs_fasta.fa")

if REFERENCE is None:
    raise FileNotFoundError(
        "Could not find Amino_acid_sequences_EL-actin_reference_clade.fa"
    )

if BIN_BY_BIN is None:
    raise FileNotFoundError("Could not find bin_by_bin/")

# Prefer the clean name, otherwise accept the original NCBI HitTable filename
BLAST_CSV = DATA / "QEE15133_viruses_blastp.csv" or DATA / "F4FTVD8P014-Alignment-HitTable.csv"

if not BLAST_CSV.exists():
    hit_tables = list(DATA.glob("*HitTable*.csv"))
    if len(hit_tables) == 1:
        BLAST_CSV = hit_tables[0]

# Keep this False for normal use.
# Set it to True only if you want the notebook to submit a remote NCBI BLAST.
RUN_REMOTE_BLAST = False

print("Actin/ARP file:", ACTIN_ARPS)
print("Reference file:", REFERENCE)
print("MAG folder:", BIN_BY_BIN)

if BLAST_CSV.exists():
    print("BLAST CSV:", BLAST_CSV)
else:
    print("BLAST CSV: not found")

# Link reference files into work/
reference_files = {
    "Actin_ARPs_fasta.fa": ACTIN_ARPS,
    "Amino_acid_sequences_EL-actin_reference_clade.fa": REFERENCE,
}

for name, src in reference_files.items():
    dst = WORK / name

    if dst.is_symlink():
        dst.unlink()

    if not dst.exists():
        dst.symlink_to(src.resolve())

# Find MAGs and link their contig files
mags = []

for contig in sorted(BIN_BY_BIN.rglob("*-contigs.fa")):
    genome = contig.name.removesuffix("-contigs.fa")

    dst = WORK / contig.name

    if dst.is_symlink():
        dst.unlink()

    if not dst.exists():
        dst.symlink_to(contig.resolve())

    mags.append(genome)

(WORK / "listeMAGs.txt").write_text(
    "\n".join(mags) + "\n"
)

IQTREE = "iqtree2" if shutil.which("iqtree2") else "iqtree"

print("WORK:", WORK)
print("MAGs:", len(mags))
print("IQ-TREE command:", IQTREE)


Actin/ARP file: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/data/actin/data_Actin/Actin_ARPs_fasta.fa
Reference file: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/data/actin/Amino_acid_sequences_EL-actin_reference_clade.fa
MAG folder: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/data/actin/01_SUMMARY_NCLDV_Genomes_with_viractin/bin_by_bin
BLAST CSV: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/data/actin/F4FTVD8P014-Alignment-HitTable.csv
WORK: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/results/actin/work
MAGs: 20
IQ-TREE command: iqtree2


## Helpers and resume controls

Set `FORCE_RERUN = False` for normal use. In this mode, completed command-line steps are **not recomputed**: the notebook reuses their output files and replays the saved terminal log when available.

Set `FORCE_RERUN = True` only when you intentionally want to recompute the command-line steps from scratch.

`REPLAY_CACHED_LOGS = True` controls whether the saved terminal output is printed when a cached result is reused.


In [2]:
# ------------------------------------------------------------------
# Resume / rerun controls
# ------------------------------------------------------------------

FORCE_RERUN = False
REPLAY_CACHED_LOGS = True

# Recovered from the previous interrupted bacterial ModelFinder run
# in this notebook. It is only used if that interrupted run left
# bacteria_model.model.gz behind but not a normal completed report/log.
RECOVERED_BACTERIA_MODEL = "LG+R7"


def remove(*names):
    for name in names:
        p = WORK / name
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists() or p.is_symlink():
            p.unlink()


def remove_glob(pattern):
    for p in WORK.glob(pattern):
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists() or p.is_symlink():
            p.unlink()


def _read_text_if_possible(path):
    try:
        if path.exists() and path.is_file():
            return path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        pass
    return ""


def _cache_available(
    expected_files=None,
    log_name=None,
    cache_regex=None,
    allow_marker_only=False
):
    expected_files = expected_files or []
    expected_paths = [WORK / name for name in expected_files]

    outputs_ok = bool(expected_paths) and all(
        p.exists() and p.stat().st_size > 0
        for p in expected_paths
    )

    if cache_regex is None:
        return outputs_ok

    texts = []

    if log_name is not None:
        texts.append(_read_text_if_possible(LOGS / log_name))

    for p in expected_paths:
        texts.append(_read_text_if_possible(p))

    marker_found = any(
        re.search(cache_regex, text, flags=re.MULTILINE)
        for text in texts
        if text
    )

    if allow_marker_only:
        return marker_found

    return outputs_ok and marker_found


def _replay_log(log_name):
    log_path = LOGS / log_name

    if not REPLAY_CACHED_LOGS:
        return

    if log_path.exists() and log_path.stat().st_size > 0:
        print("\n--- saved terminal output ---")
        text = log_path.read_text(encoding="utf-8", errors="replace")
        print(text, end="" if text.endswith("\n") else "\n")
        print("--- end saved terminal output ---\n")
    else:
        print(
            f"Saved log not found: {log_path.name} "
            "(cached result will still be reused).\n"
        )


def run(command, log_name):
    """
    Run a shell command while printing output live and writing the log
    live to disk. If a future run is interrupted, the partial log remains.
    """
    print("$", command)

    log_path = LOGS / log_name

    p = subprocess.Popen(
        command,
        cwd=WORK,
        shell=True,
        executable="/bin/bash",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    with log_path.open("w", encoding="utf-8") as log:
        for line in p.stdout:
            print(line, end="")
            log.write(line)
            log.flush()

        return_code = p.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Command failed. See results/actin/logs/{log_name}"
        )


def run_cached(
    command,
    log_name,
    expected_files=None,
    cleanup_names=None,
    cleanup_globs=None,
    cache_regex=None,
    allow_marker_only=False
):
    """
    Reuse a previous result unless FORCE_RERUN=True.

    Cached step:
        -> skip computation
        -> replay the saved terminal log

    Recomputed step:
        -> remove only that step's stale outputs
        -> run command
        -> save its log live
    """
    expected_files = expected_files or []
    cleanup_names = cleanup_names or []
    cleanup_globs = cleanup_globs or []

    cached = (
        not FORCE_RERUN
        and _cache_available(
            expected_files=expected_files,
            log_name=log_name,
            cache_regex=cache_regex,
            allow_marker_only=allow_marker_only
        )
    )

    if cached:
        print("$", command)
        print("✓ Cached result found — skipping computation.")
        _replay_log(log_name)
        return "cached"

    for name in cleanup_names:
        remove(name)

    for pattern in cleanup_globs:
        remove_glob(pattern)

    if FORCE_RERUN:
        print("↻ FORCE_RERUN=True — recomputing this step.")

    run(command, log_name)
    return "ran"


print("FORCE_RERUN =", FORCE_RERUN)
print("REPLAY_CACHED_LOGS =", REPLAY_CACHED_LOGS)


# I. Initial BlastP against Viruses

The TP starts with **Asgardactin 1 / Lokiactin 1 (QEE15133)** as the query.

The cell below downloads the protein sequence directly from NCBI if it is not already present.


In [3]:
from urllib.request import urlopen

query_file = WORK / "QEE15133.faa"

if query_file.exists():
    print("Found existing QEE15133.faa")
else:
    url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        "?db=protein&id=QEE15133&rettype=fasta&retmode=text"
    )

    sequence = urlopen(url).read()
    query_file.write_bytes(sequence)

print(query_file.read_text())


Found existing QEE15133.faa
>QEE15133.1 actin, cytoplasmic 2 [Promethearchaeum syntrophicum]
MVEEYDFLAAKPLVVDNGTGISKNGFAGEDQPRSVFPTLIGYPKYTSIMTDVEHYTREYYIGEEALQLKG
VLKLMFPVEHGVIEDWMAMEKIWHYTFYTDLRVDPSEHPVLLTEAPLNPRPNREKMAEIMFETFNVPALY
VAMQAVLSLYASGRTTGCVIDIGDGVSHVVPIYEGFALSHAISRLDLAGRDITTYLQRLLRQKGFSFTTS
AEKEIVRDIKEKLCYIALDPEKEIMLSKKVAGMEKVYMLPDGETITVSVERFLAPECFFNPSVIGKELEP
LDDVICSAIKQCDVDLRRDLYGNIVLSGGSTMYPGLKERLTKEIKEQIPESVDVKIIAPPERMYSVWIGG
SILSSLKTFQRMWVTRREYKEIGPSIIHRCF




## BlastP against Viruses

The TP restricts the search to **Viruses (taxid: 10239)**.

### Recommended

Run the search manually through the **NCBI Protein BLAST interface**, download the **Hit Table (CSV)** and place it in:

```text
data/actin/QEE15133_viruses_blastp.csv
```

This is recommended because the remote command-line BLAST can remain queued on the NCBI servers for a long time.

### Optional remote search

If you prefer to run it from the notebook, set:

```python
RUN_REMOTE_BLAST = True
```

in the setup cell.

The equivalent command is:

```bash
blastp \
-query QEE15133.faa \
-db nr \
-remote \
-entrez_query "txid10239[ORGN]" \
-out QEE15133_viruses_blastp.tsv \
-outfmt "6 qseqid saccver stitle pident qcovs evalue bitscore staxids sscinames"
```


In [4]:
remote_blast = WORK / "QEE15133_viruses_blastp.tsv"

if BLAST_CSV.exists():
    print("Using downloaded NCBI BLAST CSV:", BLAST_CSV.name)

elif remote_blast.exists():
    print("Using existing remote BLAST result:", remote_blast.name)

elif RUN_REMOTE_BLAST:
    run(
        'blastp '
        '-query QEE15133.faa '
        '-db nr '
        '-remote '
        '-entrez_query "txid10239[ORGN]" '
        '-out QEE15133_viruses_blastp.tsv '
        '-outfmt "6 qseqid saccver stitle pident qcovs evalue bitscore staxids sscinames"',
        "01_blastp_viruses.log"
    )

else:
    print(
        "No BLAST result found. Download the Hit Table CSV from NCBI "
        "into data/actin/ or set RUN_REMOTE_BLAST = True."
    )


Using downloaded NCBI BLAST CSV: F4FTVD8P014-Alignment-HitTable.csv


## BlastP results

In [5]:
remote_blast = WORK / "QEE15133_viruses_blastp.tsv"

if BLAST_CSV.exists():
    blast = pd.read_csv(BLAST_CSV)

elif remote_blast.exists():
    blast_columns = [
        "query",
        "accession",
        "title",
        "identity",
        "query_coverage",
        "evalue",
        "bitscore",
        "taxid",
        "species",
    ]

    blast = pd.read_csv(
        remote_blast,
        sep="\t",
        names=blast_columns
    )

else:
    raise FileNotFoundError(
        "No BLAST result found. Add QEE15133_viruses_blastp.csv "
        "to data/actin/ or run the optional remote BLAST."
    )

display(blast)
print("Total hits:", len(blast))


,QEE15133.1,YP_013611524.1,53.388,369,169,3,13,379,7,374,2.00e-145,425,73.44
0,QEE15133.1,P00544.1,56.923,130,54,1,13,142,30,157,9.730000e-47,176,79.23
1,QEE15133.1,CAA25063.1,56.923,130,54,1,13,142,148,275,7.560000e-46,175,79.23
2,QEE15133.1,QDH76138.1,55.462,119,51,1,20,138,1,117,1.550000e-43,155,75.63
3,QEE15133.1,QDH76139.1,57.018,114,47,1,25,138,2,113,1.640000e-42,152,78.07


Total hits: 4


# II. Identification of Viractins in NCLDV MAGs

## Predict proteins with Prodigal

```bash
while read genome; do
    prodigal -i ${genome}-contigs.fa -a ${genome}.faa
done < listeMAGs.txt
```

In [6]:
run_cached(
    'while read genome; do '
    'echo $genome; '
    'prodigal -i "${genome}-contigs.fa" -a "${genome}.faa"; '
    'done < listeMAGs.txt',
    "02_prodigal.log",
    expected_files=[f"{genome}.faa" for genome in mags],
    cleanup_names=[f"{genome}.faa" for genome in mags]
)


$ while read genome; do echo $genome; prodigal -i "${genome}-contigs.fa" -a "${genome}.faa"; done < listeMAGs.txt
Hokovirus_KY684103
-------------------------------------
PRODIGAL v2.6.3 [February, 2016]         
Univ of Tenn / Oak Ridge National Lab
Doug Hyatt, Loren Hauser, et al.     
-------------------------------------
Request:  Single Genome, Phase:  Training
Reading in the sequence(s) to train...1326289 bp seq created, 21.41 pct GC
Locating all potential starts and stops...28906 nodes
Looking for GC bias in different frames...frame bias scores: 2.55 0.28 0.17
Building initial set of genes to train from...done!
Creating coding model and scoring nodes...done!
Examining upstream regions and training starts...done!
-------------------------------------
Request:  Single Genome, Phase:  Gene Finding
Finding genes in sequence #1 (450695 bp)...done!
DEFINITION  seqnum=1;seqlen=450695;seqhdr="Hokovirus_KY684103_000000000001";gc_cont=21.41;transl_table=11;uses_sd=0
FEATURES             L

# III. Build the Actin / ARP HMM profile

## Align the Actin and ARP sequences

```bash
mafft --auto Actin_ARPs_fasta.fa > Actin_ARPs_fasta_aligned.fa
```

In [7]:
run_cached(
    "mafft --auto Actin_ARPs_fasta.fa > Actin_ARPs_fasta_aligned.fa",
    "03_mafft_hmm.log",
    expected_files=["Actin_ARPs_fasta_aligned.fa"],
    cleanup_names=["Actin_ARPs_fasta_aligned.fa"]
)


$ mafft --auto Actin_ARPs_fasta.fa > Actin_ARPs_fasta_aligned.fa
outputhat23=16
treein = 0
compacttree = 0
stacksize: 8192 kb
rescale = 1
All-to-all alignment.
    0 / 52
    1 / 52
    2 / 52
    3 / 52
    4 / 52
    5 / 52
    6 / 52
    7 / 52
    8 / 52
    9 / 52
   10 / 52
   11 / 52
   12 / 52
   13 / 52
   14 / 52
   15 / 52
   16 / 52
   17 / 52
   18 / 52
   19 / 52
   20 / 52
   21 / 52
   22 / 52
   23 / 52
   24 / 52
   25 / 52
   26 / 52
   27 / 52
   28 / 52
   29 / 52
   30 / 52
   31 / 52
   32 / 52
   33 / 52
   34 / 52
   35 / 52
   36 / 52
   37 / 52
   38 / 52
   39 / 52
   40 / 52
   41 / 52
   42 / 52
   43 / 52
   44 / 52
   45 / 52
   46 / 52
   47 / 52
   48 / 52
   49 / 52
   50 / 52
tbfast-pair (aa) Version 7.505
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a 

## Build the HMM profile

```bash
hmmbuild -n actin_like genes.hmm Actin_ARPs_fasta_aligned.fa
```

In [8]:
run_cached(
    "hmmbuild -n actin_like genes.hmm Actin_ARPs_fasta_aligned.fa",
    "04_hmmbuild.log",
    expected_files=["genes.hmm"],
    cleanup_names=["genes.hmm"]
)


$ hmmbuild -n actin_like genes.hmm Actin_ARPs_fasta_aligned.fa
# hmmbuild :: profile HMM construction from multiple sequence alignments
# HMMER 3.4 (Aug 2023); http://hmmer.org/
# Copyright (C) 2023 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# input alignment file:             Actin_ARPs_fasta_aligned.fa
# output HMM file:                  genes.hmm
# name (the single) HMM:            actin_like
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

# idx name                  nseq  alen  mlen eff_nseq re/pos description
#---- -------------------- ----- ----- ----- -------- ------ -----------
1     actin_like              52  1670   418     4.47  0.590 

# CPU time: 0.17u 0.00s 00:00:00.17 Elapsed: 00:00:00.17


# IV. Search the MAG proteins for Actin-like proteins

The non-Anvi'o route searches the **Prodigal protein files (`.faa`)**.

```bash
while read genome; do
    hmmsearch --tblout ${genome}_hmmsearch.txt genes.hmm ${genome}.faa
done < listeMAGs.txt
```

In [9]:
run_cached(
    'while read genome; do '
    'echo $genome; '
    'hmmsearch --tblout "${genome}_hmmsearch.txt" genes.hmm "${genome}.faa"; '
    'done < listeMAGs.txt',
    "05_hmmsearch.log",
    expected_files=[f"{genome}_hmmsearch.txt" for genome in mags],
    cleanup_globs=["*_hmmsearch.txt"]
)


$ while read genome; do echo $genome; hmmsearch --tblout "${genome}_hmmsearch.txt" genes.hmm "${genome}.faa"; done < listeMAGs.txt
Hokovirus_KY684103
# hmmsearch :: search profile(s) against a sequence database
# HMMER 3.4 (Aug 2023); http://hmmer.org/
# Copyright (C) 2023 Howard Hughes Medical Institute.
# Freely distributed under the BSD open source license.
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# query HMM file:                  genes.hmm
# target sequence database:        Hokovirus_KY684103.faa
# per-seq hits tabular output:     Hokovirus_KY684103_hmmsearch.txt
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

Query:       actin_like  [M=418]
Scores for complete sequences (score includes all domains):
   --- full sequence ---   --- best 1 domain ---    -#dom-
    E-value  score  bias    E-value  score  bias    exp  N  Sequence                            Description
    ------- ------ -----    ------- ------ -----   ---- -

## HMMsearch summary

In [10]:
rows = []

for genome in mags:
    result = WORK / f"{genome}_hmmsearch.txt"

    hits = []
    for line in result.read_text().splitlines():
        if line and not line.startswith("#"):
            hits.append(line.split()[0])

    rows.append({
        "MAG": genome,
        "Actin-like hits": len(set(hits))
    })

hmm_summary = pd.DataFrame(rows)
hmm_summary.to_csv(WORK / "hmmsearch_summary.csv", index=False)

display(
    hmm_summary.sort_values(
        "Actin-like hits",
        ascending=False
    ).reset_index(drop=True)
)

print("Total HMM hits:", hmm_summary["Actin-like hits"].sum())

,MAG,Actin-like hits
0,Hokovirus_KY684103,1
1,Moniruzzaman_MM01_SRX802077_122_dc,1
2,Moniruzzaman_MM15_ERX556043_42_dc,1
3,Schultz_GVMAG_M_3300003404_4,1
4,Schultz_GVMAG_M_3300009441_23,1
5,Schultz_GVMAG_M_3300020185_80,1
6,Schultz_GVMAG_M_3300020187_94,1
7,Schultz_GVMAG_M_3300021375_17,1
8,Schultz_GVMAG_M_3300024261_25,1
9,Schultz_GVMAG_M_3300023184_120,1


Total HMM hits: 19


## Extract the HMM hits

For each MAG, the hit names are read from the HMMER table and the corresponding proteins are extracted with SeqKit.

```bash
while read genome; do
    awk '$1 !~ /^#/ && NF {print $1}' ${genome}_hmmsearch.txt > ${genome}_hits.txt
    seqkit grep -f ${genome}_hits.txt ${genome}.faa
done
```

In [11]:
run_cached(
    '> tmp_viral_actin.fa; '
    'while read genome; do '
    'awk \'$1 !~ /^#/ && NF {print $1}\' "${genome}_hmmsearch.txt" '
    '| sort -u > "${genome}_hits.txt"; '
    'if [ -s "${genome}_hits.txt" ]; then '
    'seqkit grep -f "${genome}_hits.txt" "${genome}.faa" >> tmp_viral_actin.fa; '
    'fi; '
    'done < listeMAGs.txt',
    "06_extract_hmm_hits.log",
    expected_files=["tmp_viral_actin.fa"],
    cleanup_names=["tmp_viral_actin.fa"],
    cleanup_globs=["*_hits.txt"]
)

n_hits = sum(
    1 for line in open(WORK / "tmp_viral_actin.fa")
    if line.startswith(">")
)

print("Extracted Actin-like proteins:", n_hits)


$ > tmp_viral_actin.fa; while read genome; do awk '$1 !~ /^#/ && NF {print $1}' "${genome}_hmmsearch.txt" | sort -u > "${genome}_hits.txt"; if [ -s "${genome}_hits.txt" ]; then seqkit grep -f "${genome}_hits.txt" "${genome}.faa" >> tmp_viral_actin.fa; fi; done < listeMAGs.txt
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
[INFO] 1 patterns loaded from file
Extracted Actin-like proteins: 19


# V. Phylogenetic analysis of Actin / ARP diversity

## Prepare the sequence dataset

The TP asks us to:

1. remove trailing `-` characters from the reference sequences;
2. add the Viractin candidates.

```bash
seqkit seq -w 0 Amino_acid_sequences_EL-actin_reference_clade.fa \
| sed '/^>/!s/-*$//' \
> actin_reference_clean.fa

cat actin_reference_clean.fa tmp_viral_actin.fa \
> actin_viractin_dataset.fa
```

In [12]:
remove(
    "actin_reference_clean.fa",
    "actin_viractin_dataset.fa"
)

run(
    "seqkit seq -w 0 Amino_acid_sequences_EL-actin_reference_clade.fa "
    "| sed '/^>/!s/-*$//' "
    "> actin_reference_clean.fa",
    "07_clean_reference.log"
)

run(
    "cat actin_reference_clean.fa tmp_viral_actin.fa "
    "> actin_viractin_dataset.fa",
    "08_build_phylogeny_dataset.log"
)

n_sequences = sum(
    1 for line in open(WORK / "actin_viractin_dataset.fa")
    if line.startswith(">")
)

print("Sequences in phylogenetic dataset:", n_sequences)

$ seqkit seq -w 0 Amino_acid_sequences_EL-actin_reference_clade.fa | sed '/^>/!s/-*$//' > actin_reference_clean.fa
$ cat actin_reference_clean.fa tmp_viral_actin.fa > actin_viractin_dataset.fa
Sequences in phylogenetic dataset: 264


## Sequence alignment

```bash
mafft --auto actin_viractin_dataset.fa > actin_viractin_alignment.mafft.fa
```

In [13]:
run_cached(
    "mafft --auto actin_viractin_dataset.fa "
    "> actin_viractin_alignment.mafft.fa",
    "09_mafft_phylogeny.log",
    expected_files=["actin_viractin_alignment.mafft.fa"],
    cleanup_names=["actin_viractin_alignment.mafft.fa"]
)


$ mafft --auto actin_viractin_dataset.fa > actin_viractin_alignment.mafft.fa
nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8192 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 13 ambiguous characters.

    1 / 264
  101 / 264
  201 / 264
done.

Constructing a UPGMA tree (efffree=0) ... 

    0 / 264
   10 / 264
   20 / 264
   30 / 264
   40 / 264
   50 / 264
   60 / 264
   70 / 264
   80 / 264
   90 / 264
  100 / 264
  110 / 264
  120 / 264
  130 / 264
  140 / 264
  150 / 264
  160 / 264
  170 / 264
  180 / 264
  190 / 264
  200 / 264
  210 / 264
  220 / 264
  230 / 264
  240 / 264
  250 / 264
  260 / 264
done.

Progressive alignment 1/2... 

STEP     1 / 263 
STEP     2 / 263 
STEP     3 / 263 
STEP     4 / 263 
STEP     5 / 263 
STEP     6 / 263 
STEP     7 / 263 
STEP     8 / 263 
STEP     9 / 263 
STEP    10 / 263 
STEP    11 / 263 
STEP    12 / 263 
STEP    13 / 263 
STEP    14 / 263 
STEP    15 / 263 
STEP    16 / 

## Sequence trimming

```bash
trimal \
-in actin_viractin_alignment.mafft.fa \
-out actin_viractin.phy \
-phylip -gt 0.75 -cons 60
```

In [14]:
run_cached(
    "trimal "
    "-in actin_viractin_alignment.mafft.fa "
    "-out actin_viractin.phy "
    "-phylip -gt 0.75 -cons 60",
    "10_trimal.log",
    expected_files=["actin_viractin.phy"],
    cleanup_names=["actin_viractin.phy"]
)


$ trimal -in actin_viractin_alignment.mafft.fa -out actin_viractin.phy -phylip -gt 0.75 -cons 60


## Alignment before and after trimming

In [15]:
def fasta_alignment_shape(path):
    sequences = []
    current = []

    for line in path.read_text().splitlines():
        if line.startswith(">"):
            if current:
                sequences.append("".join(current))
                current = []
        else:
            current.append(line.strip())

    if current:
        sequences.append("".join(current))

    return len(sequences), len(sequences[0]) if sequences else 0


before_n, before_length = fasta_alignment_shape(
    WORK / "actin_viractin_alignment.mafft.fa"
)

phylip_header = (
    WORK / "actin_viractin.phy"
).read_text().splitlines()[0].split()

after_n = int(phylip_header[0])
after_length = int(phylip_header[1])

alignment_summary = pd.DataFrame([
    {
        "Alignment": "Before trimming",
        "Sequences": before_n,
        "Sites": before_length
    },
    {
        "Alignment": "After trimming",
        "Sequences": after_n,
        "Sites": after_length
    }
])

display(alignment_summary)

if before_length:
    print(
        "Sites retained:",
        f"{100 * after_length / before_length:.1f}%"
    )

,Alignment,Sequences,Sites
0,Before trimming,264,2921
1,After trimming,264,1753


Sites retained: 60.0%


## IQ-TREE model selection

TP command:

```bash
iqtree2 -s actin_viractin.phy -m MF -T AUTO -pre actin_model
```

The notebook uses the same parameters and stores the run under the prefix `actin_model`.

In [16]:
run_cached(
    f"{IQTREE} "
    "-s actin_viractin.phy "
    "-m MF "
    "-T AUTO "
    "-pre actin_model",
    "11_iqtree_model_selection.log",
    expected_files=["actin_model.iqtree"],
    cleanup_globs=["actin_model.*"],
    cache_regex=r"Best-fit model:.*chosen according to BIC|Best-fit model according to BIC"
)


$ iqtree2 -s actin_viractin.phy -m MF -T AUTO -pre actin_model
IQ-TREE multicore version 2.0.7 for Linux 64-bit built Apr  1 2024
Developed by Bui Quang Minh, Nguyen Lam Tung, Olga Chernomor,
Heiko Schmidt, Dominik Schrempf, Michael Woodhams.

Host:    Nora (AVX512, FMA3, 7 GB RAM)
Command: iqtree2 -s actin_viractin.phy -m MF -T AUTO -pre actin_model
Seed:    92067 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Mon Nov  9 23:43:49 2026
Kernel:  AVX+FMA - auto-detect threads (16 CPU cores detected)

Reading alignment file actin_viractin.phy ... Phylip format detected
Alignment most likely contains protein sequences
Alignment has 264 sequences with 1753 columns, 1745 distinct patterns
1053 parsimony-informative, 637 singleton sites, 63 constant sites
XP_005265644@Arp8@Homo_sapiens -> XP_005265644_Arp8_Homo_sapiens
NP_014784@Arp8p@Saccharomyces_cerevisiae_S288C -> NP_014784_Arp8p_Saccharomyces_cerevisiae_S288C
NP_573251@Arp8@Drosophila_melanogaster -> NP_573251_Arp8_Dr

## Best substitution model

In [17]:
best_model_file = WORK / "best_model.txt"

if best_model_file.exists() and not FORCE_RERUN:
    best_model = best_model_file.read_text().strip()
    print("✓ Reusing saved best model:", best_model)
else:
    sources = []

    report_file = WORK / "actin_model.iqtree"
    log_file = LOGS / "11_iqtree_model_selection.log"

    if report_file.exists():
        sources.append(report_file.read_text(
            encoding="utf-8",
            errors="replace"
        ))

    if log_file.exists():
        sources.append(log_file.read_text(
            encoding="utf-8",
            errors="replace"
        ))

    iqtree_text = "\n".join(sources)

    patterns = [
        r"Best-fit model:\s+(\S+)\s+chosen according to BIC",
        r"Best-fit model according to BIC:\s*(\S+)",
        r"Bayesian Information Criterion:\s+(\S+)"
    ]

    best_model = None

    for pattern in patterns:
        match = re.search(pattern, iqtree_text)
        if match:
            best_model = match.group(1)
            break

    if best_model is None:
        raise RuntimeError(
            "Could not automatically find the best BIC model "
            "in the IQ-TREE report or saved log."
        )

    best_model_file.write_text(best_model + "\n")

print("Best model according to BIC:", best_model)


Best model according to BIC: LG+R7


## Final phylogenetic tree

The selected model is automatically reused for the final IQ-TREE run.

Equivalent TP command:

```bash
iqtree2 \
-s actin_viractin.phy \
-m BEST_MODEL \
-alrt 1000 \
-B 1000 \
-T AUTO \
-pre actin_tree
```

In [18]:
best_model = (WORK / "best_model.txt").read_text().strip()

run_cached(
    f"{IQTREE} "
    "-s actin_viractin.phy "
    f"-m {best_model} "
    "-alrt 1000 "
    "-B 1000 "
    "-T AUTO "
    "-pre actin_tree",
    "12_iqtree_final_tree.log",
    expected_files=["actin_tree.treefile"],
    cleanup_globs=["actin_tree.*"]
)

print("Final tree:", WORK / "actin_tree.treefile")


$ iqtree2 -s actin_viractin.phy -m LG+R7 -alrt 1000 -B 1000 -T AUTO -pre actin_tree
IQ-TREE multicore version 2.0.7 for Linux 64-bit built Apr  1 2024
Developed by Bui Quang Minh, Nguyen Lam Tung, Olga Chernomor,
Heiko Schmidt, Dominik Schrempf, Michael Woodhams.

Host:    Nora (AVX512, FMA3, 7 GB RAM)
Command: iqtree2 -s actin_viractin.phy -m LG+R7 -alrt 1000 -B 1000 -T AUTO -pre actin_tree
Seed:    237343 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Nov 10 02:20:19 2026
Kernel:  AVX+FMA - auto-detect threads (16 CPU cores detected)

Reading alignment file actin_viractin.phy ... Phylip format detected
Alignment most likely contains protein sequences
Alignment has 264 sequences with 1753 columns, 1745 distinct patterns
1053 parsimony-informative, 637 singleton sites, 63 constant sites
XP_005265644@Arp8@Homo_sapiens -> XP_005265644_Arp8_Homo_sapiens
NP_014784@Arp8p@Saccharomyces_cerevisiae_S288C -> NP_014784_Arp8p_Saccharomyces_cerevisiae_S288C
NP_573251@Arp8@D

# VI. Tree visualization

Upload the final tree to **iTOL**:

```text
results/actin/work/actin_tree.treefile
```

Then add the explanatory layer requested in the TP:

- use the reference sequence groups from Table S1;
- label the NCLDV sequences as **Viractin**.


In [19]:
tree = WORK / "actin_tree.treefile"

if tree.exists():
    display(
        FileLink(
            str(tree.relative_to(ROOT)),
            result_html_prefix="Final Actin / Viractin tree: "
        )
    )

/home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/metagenomics-diversity-analysis/results/actin/work/actin_tree.treefile

# VII. To go further — add the Bacterial Actin-like proteins

The TP asks us to add:

- `RKY65704.1`
- `WP_173013208.1`

The cell below downloads both proteins directly from NCBI.

In [20]:
from urllib.parse import urlencode

remove("bacterial_actins.fa")

params = urlencode({
    "db": "protein",
    "id": "RKY65704.1,WP_173013208.1",
    "rettype": "fasta",
    "retmode": "text",
})

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?"
    + params
)

sequences = urlopen(url).read()

(WORK / "bacterial_actins.fa").write_bytes(sequences)

print((WORK / "bacterial_actins.fa").read_text())

>RKY65704.1 MAG: actin, cytoplasmic 2 [Candidatus Latescibacterota bacterium]
MAEPATPEEWLAKQPIVIDNGTGLTKNGFAGEDNPRSVWPSLIGYPKYRAIMSDVDHYQREYYIGKEALN
LKGVLKLVYPIEHGIINDWKAMERIWHYTFFNDLKVDPSQHPVLLTEPPMNPVSNREKMAEIMFETFGVP
AIYVSMQAVLSLYASGRTTGLVIDSGDGVTHIVPIFEGFAITHAIRRIDLGGRDITEFLRRLLRQRGYTL
VTSAEREIVRDIKETLTYVALDPEKEWDLYKKMGDKMEKSYTLPDGTVIRIGPERFQAPEVFFNPSLIGL
EESPLDEVIVDVVQACDVDLRRDLYANIVLSGGSTMFPGLKERLYKEIKELVPESVEVKIIAPPERQYSV
WIGGSILASLKTFQRMWVTRKEYDEQGPAAVRRCF

>WP_173013208.1 actin family protein [Candidatus Uabimicrobium amorphum]
MKNLKKLCFIMAMLVMASLQAEKSAIVIYIDSFVLHAGFAGDDAPKIVMPNAVSTKDDEVYVDFGTLAKS
DSVSYPIQDGKVVDVAQCEKVLDHVFSKLKADPKEQAVLLAYPGDFTTESIDQVAKILFDKFEVPALHAV
SNTILILYSNGKMDGLVVCMDDNGVFVNAVDYGSILATQKIPVTSQHIRDYLQKLLEKKGYQFAKNCSGM
MEEIKNKYCYVALNFNAETKLAKNSSSLSKTYTLEDGQAIEVKEERFTAPEAFFQPKLLGIDAQGVHKVI
QEVIGKCDSKFRKRLYRNIVIAGSESMYPGLRDRIEYEFYNANKKMKTKVSAPPERKYGTWIGGSIFASL
SNFKSVPRKEYMDHGSFGDLQKKLEQK




## Add the Bacterial proteins

```bash
cat actin_viractin_dataset.fa bacterial_actins.fa \
> actin_viractin_bacteria_dataset.fa
```

In [21]:
remove("actin_viractin_bacteria_dataset.fa")

run(
    "cat actin_viractin_dataset.fa bacterial_actins.fa "
    "> actin_viractin_bacteria_dataset.fa",
    "13_add_bacterial_sequences.log"
)

$ cat actin_viractin_dataset.fa bacterial_actins.fa > actin_viractin_bacteria_dataset.fa


## Align the extended dataset

```bash
mafft --auto actin_viractin_bacteria_dataset.fa \
> actin_viractin_bacteria_alignment.mafft.fa
```

In [22]:
run_cached(
    "mafft --auto actin_viractin_bacteria_dataset.fa "
    "> actin_viractin_bacteria_alignment.mafft.fa",
    "14_mafft_bacteria.log",
    expected_files=["actin_viractin_bacteria_alignment.mafft.fa"],
    cleanup_names=["actin_viractin_bacteria_alignment.mafft.fa"]
)


$ mafft --auto actin_viractin_bacteria_dataset.fa > actin_viractin_bacteria_alignment.mafft.fa
nthread = 0
nthreadpair = 0
nthreadtb = 0
ppenalty_ex = 0
stacksize: 8192 kb
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..

There are 13 ambiguous characters.

    1 / 266
  101 / 266
  201 / 266
done.

Constructing a UPGMA tree (efffree=0) ... 

    0 / 266
   10 / 266
   20 / 266
   30 / 266
   40 / 266
   50 / 266
   60 / 266
   70 / 266
   80 / 266
   90 / 266
  100 / 266
  110 / 266
  120 / 266
  130 / 266
  140 / 266
  150 / 266
  160 / 266
  170 / 266
  180 / 266
  190 / 266
  200 / 266
  210 / 266
  220 / 266
  230 / 266
  240 / 266
  250 / 266
  260 / 266
done.

Progressive alignment 1/2... 

STEP     1 / 265 
STEP     2 / 265 
STEP     3 / 265 
STEP     4 / 265 
STEP     5 / 265 
STEP     6 / 265 
STEP     7 / 265 
STEP     8 / 265 
STEP     9 / 265 
STEP    10 / 265 
STEP    11 / 265 
STEP    12 / 265 
STEP    13 / 265 
STEP    14 / 265 
STEP    15 / 

## Trim the extended alignment

```bash
trimal \
-in actin_viractin_bacteria_alignment.mafft.fa \
-out actin_viractin_bacteria.phy \
-phylip -gt 0.75 -cons 60
```

In [23]:
run_cached(
    "trimal "
    "-in actin_viractin_bacteria_alignment.mafft.fa "
    "-out actin_viractin_bacteria.phy "
    "-phylip -gt 0.75 -cons 60",
    "15_trimal_bacteria.log",
    expected_files=["actin_viractin_bacteria.phy"],
    cleanup_names=["actin_viractin_bacteria.phy"]
)


$ trimal -in actin_viractin_bacteria_alignment.mafft.fa -out actin_viractin_bacteria.phy -phylip -gt 0.75 -cons 60


## IQ-TREE model selection

```bash
iqtree2 \
-s actin_viractin_bacteria.phy \
-m MF \
-T AUTO \
-pre bacteria_model
```

In [ ]:
bacteria_model_regex = (
    r"Best-fit model:.*chosen according to BIC|"
    r"Bayesian Information Criterion:\s+\S+"
)

bacteria_model_cached = _cache_available(
    expected_files=["bacteria_model.iqtree"],
    log_name="16_iqtree_bacteria_model.log",
    cache_regex=bacteria_model_regex,
    allow_marker_only=True
)

interrupted_model_artifact = WORK / "bacteria_model.model.gz"

if (
    not FORCE_RERUN
    and not bacteria_model_cached
    and interrupted_model_artifact.exists()
    and interrupted_model_artifact.stat().st_size > 0
):
    print(
        "✓ Previous interrupted ModelFinder run detected "
        "(bacteria_model.model.gz exists)."
    )
    print(
        "✓ Reusing the BIC-selected model recovered from that run:",
        RECOVERED_BACTERIA_MODEL
    )

    (WORK / "best_bacteria_model.txt").write_text(
        RECOVERED_BACTERIA_MODEL + "\n"
    )

    _replay_log("16_iqtree_bacteria_model.log")
else:
    run_cached(
        f"{IQTREE} "
        "-s actin_viractin_bacteria.phy "
        "-m MF "
        "-T AUTO "
        "-pre bacteria_model",
        "16_iqtree_bacteria_model.log",
        expected_files=["bacteria_model.iqtree"],
        cleanup_globs=["bacteria_model.*"],
        cache_regex=bacteria_model_regex,
        allow_marker_only=True
    )


$ iqtree2 -s actin_viractin_bacteria.phy -m MF -T AUTO -pre bacteria_model
IQ-TREE multicore version 2.0.7 for Linux 64-bit built Apr  1 2024
Developed by Bui Quang Minh, Nguyen Lam Tung, Olga Chernomor,
Heiko Schmidt, Dominik Schrempf, Michael Woodhams.

Host:    Nora (AVX512, FMA3, 7 GB RAM)
Command: iqtree2 -s actin_viractin_bacteria.phy -m MF -T AUTO -pre bacteria_model
Seed:    988254 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Tue Nov 10 08:55:23 2026
Kernel:  AVX+FMA - auto-detect threads (16 CPU cores detected)

Reading alignment file actin_viractin_bacteria.phy ... Phylip format detected
Alignment most likely contains protein sequences
Alignment has 266 sequences with 1755 columns, 1747 distinct patterns
1063 parsimony-informative, 646 singleton sites, 46 constant sites
XP_005265644@Arp8@Homo_sapiens -> XP_005265644_Arp8_Homo_sapiens
NP_014784@Arp8p@Saccharomyces_cerevisiae_S288C -> NP_014784_Arp8p_Saccharomyces_cerevisiae_S288C
NP_573251@Arp8@Drosophila

## Best model for the extended dataset

In [ ]:
best_bacteria_model_file = WORK / "best_bacteria_model.txt"

if best_bacteria_model_file.exists() and not FORCE_RERUN:
    best_bacteria_model = (
        best_bacteria_model_file.read_text().strip()
    )
    print(
        "✓ Reusing saved bacterial best model:",
        best_bacteria_model
    )
else:
    sources = []

    report_file = WORK / "bacteria_model.iqtree"
    log_file = LOGS / "16_iqtree_bacteria_model.log"

    if report_file.exists():
        sources.append(report_file.read_text(
            encoding="utf-8",
            errors="replace"
        ))

    if log_file.exists():
        sources.append(log_file.read_text(
            encoding="utf-8",
            errors="replace"
        ))

    iqtree_text = "\n".join(sources)

    patterns = [
        r"Best-fit model:\s+(\S+)\s+chosen according to BIC",
        r"Best-fit model according to BIC:\s*(\S+)",
        r"Bayesian Information Criterion:\s+(\S+)"
    ]

    best_bacteria_model = None

    for pattern in patterns:
        match = re.search(pattern, iqtree_text)
        if match:
            best_bacteria_model = match.group(1)
            break

    if (
        best_bacteria_model is None
        and not FORCE_RERUN
        and (WORK / "bacteria_model.model.gz").exists()
    ):
        best_bacteria_model = RECOVERED_BACTERIA_MODEL
        print(
            "✓ Recovered bacterial model from the previous "
            "interrupted run:",
            best_bacteria_model
        )

    if best_bacteria_model is None:
        raise RuntimeError(
            "Could not automatically find the best BIC model "
            "in bacteria_model.iqtree or the saved log."
        )

    best_bacteria_model_file.write_text(
        best_bacteria_model + "\n"
    )

print(
    "Best model according to BIC:",
    best_bacteria_model
)


## Final tree including Bacteria

```bash
iqtree2 -s actin_viractin_bacteria.phy \
  -m LG+R7 \
  -alrt 1000 \
  -B 1000 \
  -T AUTO \
  -pre bacteria_tree
```

In [ ]:
best_bacteria_model = (
    WORK / "best_bacteria_model.txt"
).read_text().strip()

run_cached(
    f"{IQTREE} "
    "-s actin_viractin_bacteria.phy "
    f"-m {best_bacteria_model} "
    "-alrt 1000 "
    "-B 1000 "
    "-T AUTO "
    "-pre bacteria_tree",
    "17_iqtree_bacterial_final_tree.log",
    expected_files=["bacteria_tree.treefile"],
    cleanup_globs=["bacteria_tree.*"]
)

tree_file = WORK / "bacteria_tree.treefile"

if not tree_file.exists():
    raise RuntimeError("Final bacterial tree was not generated.")

print(
    "Final tree including Bacteria:",
    tree_file
)


# Outputs

Main analysis files are in:

```text
results/actin/work/
```

Important outputs:

```text
hmmsearch_summary.csv
tmp_viral_actin.fa
actin_viractin_alignment.mafft.fa
actin_viractin.phy
best_model.txt
actin_tree.treefile
bacteria_tree.treefile
```

The manually downloaded NCBI BlastP Hit Table stays in:

```text
data/actin/QEE15133_viruses_blastp.csv
```

Terminal output logs are in:

```text
results/actin/logs/
```


## Resume behavior

At the top of the notebook:

```python
FORCE_RERUN = False
REPLAY_CACHED_LOGS = True
```

- `FORCE_RERUN = False`: reuse existing outputs and replay saved logs.
- `FORCE_RERUN = True`: delete the relevant cached outputs and recompute command-line steps.
- `REPLAY_CACHED_LOGS = False`: reuse cached outputs without printing the old terminal log.

Logs are now written **while commands are running**, so even an interrupted future run leaves a partial log on disk.
